# Turn-Level Dynamics: Cognitive Endurance Analysis

## Core Question
**Does the communication bandwidth degrade as the game progresses?**

## Hypothesis: The "Low-Hanging Fruit" Effect
Codenames gets harder in later turns because obvious semantic clusters (e.g., "Apple", "Banana" → FRUIT 2) are cleared early. Late-game clues require more abstract, subtle lateral thinking.

**Prediction:**
- **Different Persona** pairs will see their performance crash in late turns (divergence)
- **Same Persona** pairs will maintain their efficiency (robustness)

## Metric: Clue Utilization
$$\text{Clue Utilization} = \frac{\text{correct\_guesses}}{\text{clue\_number}}$$

This normalizes data regardless of how "ambitious" the move was.

## Data Processing
- Group by turn_number (1, 2, 3, ...)
- Split into Same Persona (CM ID == Guesser ID) vs Different Persona
- Filter: Only turns where clue_number > 0
- Truncate at Turn 6 (most winning games end by then)

## 1. Setup and Imports

In [ ]:
import json
import os
import glob
import re
from collections import defaultdict
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import ttest_ind, mannwhitneyu

# Import unified loading functions
from analysis_utils import load_turn_level_data

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Plotting style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# ===== LOGGING SETUP =====
LOG_FILE = 'turn_level_dynamics.txt'

def clear_log():
    """Clear the log file at start of notebook run."""
    with open(LOG_FILE, 'w') as f:
        f.write("=" * 80 + "\n")
        f.write("TURN-LEVEL DYNAMICS: COGNITIVE ENDURANCE ANALYSIS\n")
        f.write("=" * 80 + "\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        f.write("This file contains all numerical results and statistics from turn_level_dynamics.ipynb.\n")
        f.write("For plots and visualizations, see the figures/ directory.\n")
        f.write("=" * 80 + "\n\n")

def log_print(*args, sep=' ', end='\n'):
    """Print to console and append to log file."""
    text = sep.join(str(arg) for arg in args)
    print(text, end=end)
    with open(LOG_FILE, 'a') as f:
        f.write(text + end)

# Clear log file at notebook start
clear_log()

# ===== CONFIGURATION =====
MAX_PERSONA_ID = 5  # Set to match your experiment configuration
LOGS_DIR = 'game_logs/tailored_words_to_persona1'  # Directory containing game logs
MAX_TURN = 10  # Truncate analysis at this turn (avoid "zombie games")

log_print(f"Analysis configured for personas 1-{MAX_PERSONA_ID}")
log_print(f"Loading logs from: {LOGS_DIR}")
log_print(f"Truncating analysis at Turn {MAX_TURN}")

## 2. Load and Parse Game Logs

In [ ]:
# parse_game_filename is now provided by analysis_utils module
# This cell is kept for backward compatibility but the function is imported

In [ ]:
# Load data using unified loading function from analysis_utils
# This function auto-detects the log format (old or new) and handles both

turns_df = load_turn_level_data(LOGS_DIR, MAX_PERSONA_ID)
log_print(f"\nTurns shape: {turns_df.shape}")
turns_df.head(10)

In [ ]:
# Summary statistics
log_print("\n" + "=" * 80)
log_print("DATA SUMMARY")
log_print("=" * 80)

# Count by cohort
cohort_counts = turns_df.groupby('cohort').agg({
    'game_id': 'nunique',
    'turn_number': 'count'
}).rename(columns={'game_id': 'unique_games', 'turn_number': 'total_turns'})

log_print("\n=== Cohort Breakdown ===")
for cohort, row in cohort_counts.iterrows():
    log_print(f"  {cohort}: {int(row['unique_games'])} games, {int(row['total_turns'])} turns")

# Unique persona pairs
same_pairs = turns_df[turns_df['is_same_persona']].groupby(['cm_id', 'guesser_id']).size()
diff_pairs = turns_df[~turns_df['is_same_persona']].groupby(['cm_id', 'guesser_id']).size()

log_print(f"\n=== Unique Persona Pairs ===")
log_print(f"  Same Persona pairs: {len(same_pairs)} (n={len(same_pairs)})")
log_print(f"  Different Persona pairs: {len(diff_pairs)} (n={len(diff_pairs)})")

# Turn distribution
log_print(f"\n=== Turn Distribution ===")
turn_dist = turns_df['turn_number'].value_counts().sort_index()
for turn_num, count in turn_dist.items():
    log_print(f"  Turn {turn_num}: {count} observations")

## 3. Compute Mean Clue Utilization per Turn

In [ ]:
def compute_turn_statistics(df, max_turn=6):
    """
    Compute mean clue utilization and confidence intervals per turn for each cohort.
    
    Returns:
        stats_df: DataFrame with turn-level statistics
    """
    # Filter to max_turn
    df_filtered = df[df['turn_number'] <= max_turn].copy()
    
    stats_records = []
    
    for cohort in ['Same Persona', 'Different Persona']:
        cohort_df = df_filtered[df_filtered['cohort'] == cohort]
        
        for turn_num in range(1, max_turn + 1):
            turn_data = cohort_df[cohort_df['turn_number'] == turn_num]['clue_utilization']
            
            n = len(turn_data)
            if n == 0:
                continue
            
            mean_util = turn_data.mean()
            std_util = turn_data.std()
            sem = std_util / np.sqrt(n)  # Standard Error of the Mean
            
            # 95% Confidence Interval
            ci_95 = 1.96 * sem
            ci_lower = mean_util - ci_95
            ci_upper = mean_util + ci_95
            
            stats_records.append({
                'cohort': cohort,
                'turn_number': turn_num,
                'n': n,
                'mean_clue_util': mean_util,
                'std': std_util,
                'sem': sem,
                'ci_95': ci_95,
                'ci_lower': ci_lower,
                'ci_upper': ci_upper
            })
    
    return pd.DataFrame(stats_records)


# Compute statistics
turn_stats = compute_turn_statistics(turns_df, max_turn=MAX_TURN)

log_print("\n" + "=" * 80)
log_print("TURN-LEVEL CLUE UTILIZATION STATISTICS")
log_print("=" * 80)

log_print("\n=== Same Persona ===")
same_stats = turn_stats[turn_stats['cohort'] == 'Same Persona']
for _, row in same_stats.iterrows():
    log_print(f"  Turn {int(row['turn_number'])}: Mean={row['mean_clue_util']:.3f} ± {row['ci_95']:.3f} (n={int(row['n'])})")

log_print("\n=== Different Persona ===")
diff_stats = turn_stats[turn_stats['cohort'] == 'Different Persona']
for _, row in diff_stats.iterrows():
    log_print(f"  Turn {int(row['turn_number'])}: Mean={row['mean_clue_util']:.3f} ± {row['ci_95']:.3f} (n={int(row['n'])})")

turn_stats

In [ ]:
# Check if sample sizes are adequate
log_print("\n=== Sample Size Check ===")
log_print("(Checking if data gets too sparse by Turn 6)\n")

min_samples = turn_stats.groupby('cohort')['n'].min()
for cohort, min_n in min_samples.items():
    status = "✓ Adequate" if min_n >= 20 else "⚠ Sparse (< 20 samples)"
    log_print(f"  {cohort}: Minimum n = {min_n} {status}")

# Show the sparse turns if any
sparse_turns = turn_stats[turn_stats['n'] < 20]
if len(sparse_turns) > 0:
    log_print("\n  Sparse turns (n < 20):")
    for _, row in sparse_turns.iterrows():
        log_print(f"    {row['cohort']}, Turn {int(row['turn_number'])}: n = {int(row['n'])}")

## 4. Statistical Tests: Turn-by-Turn Comparison

In [ ]:
# Perform turn-by-turn statistical tests
log_print("\n" + "=" * 80)
log_print("STATISTICAL TESTS: SAME vs DIFFERENT PERSONA BY TURN")
log_print("=" * 80)
log_print("\nTesting whether Same Persona pairs have significantly higher")
log_print("clue utilization than Different Persona pairs at each turn.\n")
log_print("Significance levels: *** p<0.001, ** p<0.01, * p<0.05, (n.s.) not significant\n")

test_results = []
turns_filtered = turns_df[turns_df['turn_number'] <= MAX_TURN]

for turn_num in range(1, MAX_TURN + 1):
    same_data = turns_filtered[(turns_filtered['turn_number'] == turn_num) & 
                                (turns_filtered['is_same_persona'])]['clue_utilization']
    diff_data = turns_filtered[(turns_filtered['turn_number'] == turn_num) & 
                                (~turns_filtered['is_same_persona'])]['clue_utilization']
    
    if len(same_data) < 3 or len(diff_data) < 3:
        log_print(f"Turn {turn_num}: Insufficient data for statistical test")
        continue
    
    # Independent t-test
    t_stat, p_value = ttest_ind(same_data, diff_data)
    
    # Mann-Whitney U test (non-parametric alternative)
    u_stat, u_p_value = mannwhitneyu(same_data, diff_data, alternative='two-sided')
    
    # Significance markers
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else '(n.s.)'
    
    # Effect size (Cohen's d)
    pooled_std = np.sqrt(((len(same_data)-1)*same_data.std()**2 + (len(diff_data)-1)*diff_data.std()**2) / 
                         (len(same_data) + len(diff_data) - 2))
    cohens_d = (same_data.mean() - diff_data.mean()) / pooled_std if pooled_std > 0 else 0
    
    test_results.append({
        'turn': turn_num,
        'same_mean': same_data.mean(),
        'diff_mean': diff_data.mean(),
        'same_n': len(same_data),
        'diff_n': len(diff_data),
        't_stat': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'sig': sig
    })
    
    delta = same_data.mean() - diff_data.mean()
    direction = "Same > Diff" if delta > 0 else "Diff > Same"
    log_print(f"Turn {turn_num}: Same={same_data.mean():.3f} (n={len(same_data)}), "
              f"Diff={diff_data.mean():.3f} (n={len(diff_data)})")
    log_print(f"         Δ = {delta:+.3f} ({direction}), t = {t_stat:.2f}, p = {p_value:.4f} {sig}")
    log_print(f"         Cohen's d = {cohens_d:.3f}\n")

test_results_df = pd.DataFrame(test_results)
test_results_df

## 5. Main Visualization: Cognitive Endurance Plot

In [ ]:
# Create the main visualization: Line plot with confidence intervals
fig, ax = plt.subplots(figsize=(10, 7))

# Prepare data for plotting
same_stats = turn_stats[turn_stats['cohort'] == 'Same Persona'].sort_values('turn_number')
diff_stats = turn_stats[turn_stats['cohort'] == 'Different Persona'].sort_values('turn_number')

# Plot Same Persona (Red, Solid)
ax.plot(same_stats['turn_number'], same_stats['mean_clue_util'], 
        color='red', linewidth=2.5, marker='o', markersize=8,
        label=f'Same Persona (n={len(same_pairs)} pairs)')
ax.fill_between(same_stats['turn_number'], 
                same_stats['ci_lower'], 
                same_stats['ci_upper'],
                color='red', alpha=0.2)

# Plot Different Persona (Blue, Dashed)
ax.plot(diff_stats['turn_number'], diff_stats['mean_clue_util'], 
        color='blue', linewidth=2.5, linestyle='--', marker='s', markersize=8,
        label=f'Different Persona (n={len(diff_pairs)} pairs)')
ax.fill_between(diff_stats['turn_number'], 
                diff_stats['ci_lower'], 
                diff_stats['ci_upper'],
                color='blue', alpha=0.2)

# Formatting
ax.set_xlabel('Game Turn', fontsize=14)
ax.set_ylabel('Mean Clue Utilization', fontsize=14)
ax.set_title('Cognitive Endurance: Does Communication Bandwidth Degrade Over Turns?', 
             fontsize=14, fontweight='bold', pad=20)

ax.set_xlim(0.5, MAX_TURN + 0.5)
ax.set_ylim(0, 1.0)
ax.set_xticks(range(1, MAX_TURN + 1))

ax.legend(loc='upper right', fontsize=11)
ax.grid(True, alpha=0.3)

# Add annotations for significant turns
for _, row in test_results_df.iterrows():
    if row['sig'] != '(n.s.)':
        # Find y-position (midpoint between the two lines)
        y_pos = max(row['same_mean'], row['diff_mean']) + 0.05
        ax.annotate(row['sig'], xy=(row['turn'], y_pos), 
                   fontsize=12, ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('figures/cognitive_endurance_plot.pdf', dpi=300, bbox_inches='tight')
plt.show()

log_print("\n" + "=" * 80)
log_print("MAIN VISUALIZATION SAVED")
log_print("=" * 80)
log_print("\nPlot saved to: figures/cognitive_endurance_plot.pdf")

## 6. Detailed Analysis: The "Divergence" Story

In [ ]:
# Analyze the divergence pattern
log_print("\n" + "=" * 80)
log_print("DIVERGENCE ANALYSIS: THE LOW-HANGING FRUIT HYPOTHESIS")
log_print("=" * 80)

# Calculate slopes (linear regression for each cohort)
from scipy.stats import linregress

log_print("\n=== Trend Analysis (Linear Regression) ===")
log_print("Testing whether clue utilization declines over turns.\n")

for cohort_name, cohort_label in [('Same Persona', 'Same'), ('Different Persona', 'Different')]:
    cohort_stats = turn_stats[turn_stats['cohort'] == cohort_name]
    
    if len(cohort_stats) < 2:
        continue
    
    slope, intercept, r_value, p_value, std_err = linregress(
        cohort_stats['turn_number'], 
        cohort_stats['mean_clue_util']
    )
    
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else '(n.s.)'
    
    log_print(f"{cohort_label} Persona:")
    log_print(f"  Slope = {slope:.4f} (change per turn)")
    log_print(f"  R² = {r_value**2:.4f}")
    log_print(f"  p-value = {p_value:.4f} {sig}")
    
    if slope < 0:
        log_print(f"  → Clue utilization DECREASES by {abs(slope)*100:.1f}% per turn")
    else:
        log_print(f"  → Clue utilization INCREASES by {slope*100:.1f}% per turn")
    log_print("")

In [ ]:
# Phase analysis: Early vs Late game
log_print("\n=== Phase Analysis: Early Game (Turns 1-2) vs Late Game (Turns 4-6) ===")
log_print("\nComparing performance in early vs late turns by cohort.\n")

turns_filtered = turns_df[turns_df['turn_number'] <= MAX_TURN].copy()
turns_filtered['phase'] = turns_filtered['turn_number'].apply(
    lambda x: 'Early (1-2)' if x <= 2 else 'Late (4-6)' if x >= 4 else 'Mid (3)'
)

# Focus on Early vs Late
early_late = turns_filtered[turns_filtered['phase'].isin(['Early (1-2)', 'Late (4-6)'])]

phase_stats = early_late.groupby(['cohort', 'phase'])['clue_utilization'].agg(['mean', 'std', 'count'])
phase_stats.columns = ['mean', 'std', 'n']

log_print("Performance by Phase:")
for idx, row in phase_stats.iterrows():
    log_print(f"  {idx[0]}, {idx[1]}: Mean = {row['mean']:.3f} ± {row['std']:.3f} (n={int(row['n'])})")

# Compute drop for each cohort
log_print("\n=== Performance Drop (Early → Late) ===")
for cohort in ['Same Persona', 'Different Persona']:
    try:
        early_mean = phase_stats.loc[(cohort, 'Early (1-2)'), 'mean']
        late_mean = phase_stats.loc[(cohort, 'Late (4-6)'), 'mean']
        drop = early_mean - late_mean
        pct_drop = (drop / early_mean) * 100
        
        log_print(f"  {cohort}: {early_mean:.3f} → {late_mean:.3f} (Δ = {drop:+.3f}, {pct_drop:+.1f}% change)")
    except KeyError:
        log_print(f"  {cohort}: Insufficient data for comparison")

In [ ]:
# Compute "Robustness" metric: standard deviation of clue utilization across turns
log_print("\n=== Robustness Analysis ===")
log_print("(Lower variance across turns = more consistent performance)\n")

# For each game, compute the variance of clue utilization across turns
game_variance = turns_filtered.groupby(['game_id', 'cohort'])['clue_utilization'].agg(['mean', 'std', 'count'])
game_variance.columns = ['mean_util', 'std_util', 'num_turns']
game_variance = game_variance.reset_index()

# Filter games with at least 3 turns for meaningful variance
game_variance = game_variance[game_variance['num_turns'] >= 3]

# Compare variance between cohorts
same_var = game_variance[game_variance['cohort'] == 'Same Persona']['std_util'].dropna()
diff_var = game_variance[game_variance['cohort'] == 'Different Persona']['std_util'].dropna()

log_print(f"Same Persona games (n={len(same_var)}):")
log_print(f"  Mean within-game std: {same_var.mean():.3f} ± {same_var.std():.3f}")

log_print(f"\nDifferent Persona games (n={len(diff_var)}):")
log_print(f"  Mean within-game std: {diff_var.mean():.3f} ± {diff_var.std():.3f}")

# Statistical test
if len(same_var) >= 3 and len(diff_var) >= 3:
    t_stat, p_value = ttest_ind(same_var, diff_var)
    sig = '***' if p_value < 0.001 else '**' if p_value < 0.01 else '*' if p_value < 0.05 else '(n.s.)'
    
    log_print(f"\nStatistical Test (t-test on within-game std):")
    log_print(f"  t = {t_stat:.3f}, p = {p_value:.4f} {sig}")
    
    if p_value < 0.05:
        more_robust = "Same Persona" if same_var.mean() < diff_var.mean() else "Different Persona"
        log_print(f"  → {more_robust} pairs are significantly more consistent across turns")
    else:
        log_print(f"  → No significant difference in consistency")

## 7. Supplementary Visualization: Detailed Breakdowns

In [ ]:
# Violin plot showing distribution by turn and cohort
fig, ax = plt.subplots(figsize=(14, 6))

turns_plot = turns_df[turns_df['turn_number'] <= MAX_TURN].copy()

sns.violinplot(
    data=turns_plot,
    x='turn_number',
    y='clue_utilization',
    hue='cohort',
    split=True,
    palette={'Same Persona': 'red', 'Different Persona': 'blue'},
    ax=ax,
    inner='quartile'
)

ax.set_xlabel('Game Turn', fontsize=14)
ax.set_ylabel('Clue Utilization', fontsize=14)
ax.set_title('Distribution of Clue Utilization by Turn and Persona Type', fontsize=14, fontweight='bold')
ax.legend(title='Cohort', loc='upper right')
ax.set_ylim(-0.1, 1.6)

plt.tight_layout()
plt.savefig('figures/cognitive_endurance_violin.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Heatmap: Clue utilization by persona pair and turn
# Focus on same-persona pairs to see if some maintain better than others

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Same Persona heatmap
same_turns = turns_df[(turns_df['is_same_persona']) & (turns_df['turn_number'] <= MAX_TURN)]
same_pivot = same_turns.pivot_table(
    values='clue_utilization',
    index='cm_id',
    columns='turn_number',
    aggfunc='mean'
)

sns.heatmap(
    same_pivot,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    vmin=0,
    vmax=1,
    ax=axes[0],
    cbar_kws={'label': 'Clue Utilization'}
)
axes[0].set_title('Same Persona Pairs: Utilization by Turn', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Turn Number', fontsize=11)
axes[0].set_ylabel('Persona ID', fontsize=11)

# Overall average by turn (Different Persona)
diff_turns = turns_df[(~turns_df['is_same_persona']) & (turns_df['turn_number'] <= MAX_TURN)]
diff_avg_by_turn = diff_turns.groupby('turn_number')['clue_utilization'].agg(['mean', 'std', 'count'])

axes[1].bar(diff_avg_by_turn.index, diff_avg_by_turn['mean'], 
            yerr=diff_avg_by_turn['std']/np.sqrt(diff_avg_by_turn['count']),
            capsize=5, color='steelblue', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Turn Number', fontsize=11)
axes[1].set_ylabel('Mean Clue Utilization', fontsize=11)
axes[1].set_title('Different Persona Pairs: Average by Turn', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 0.8)
axes[1].grid(True, alpha=0.3, axis='y')

# Add sample size annotations
for i, (turn, row) in enumerate(diff_avg_by_turn.iterrows()):
    axes[1].annotate(f'n={int(row["count"])}', xy=(turn, row['mean'] + 0.05), 
                    ha='center', fontsize=9, color='gray')

plt.tight_layout()
plt.savefig('figures/cognitive_endurance_details.pdf', dpi=300, bbox_inches='tight')
plt.show()

## 8. Summary and Interpretation

In [ ]:
log_print("\n" + "=" * 80)
log_print("SUMMARY: COGNITIVE ENDURANCE ANALYSIS")
log_print("=" * 80)

log_print("\n=== Key Findings ===")

# 1. Overall comparison
same_overall = turns_df[turns_df['is_same_persona']]['clue_utilization'].mean()
diff_overall = turns_df[~turns_df['is_same_persona']]['clue_utilization'].mean()
log_print(f"\n1. OVERALL CLUE UTILIZATION:")
log_print(f"   Same Persona: {same_overall:.3f}")
log_print(f"   Different Persona: {diff_overall:.3f}")
log_print(f"   Δ = {same_overall - diff_overall:+.3f} ({'Same > Diff' if same_overall > diff_overall else 'Diff > Same'})")

# 2. Trend summary
log_print(f"\n2. DEGRADATION OVER TURNS:")
if len(same_stats) >= 2 and len(diff_stats) >= 2:
    same_t1 = same_stats[same_stats['turn_number'] == 1]['mean_clue_util'].values[0]
    same_t6 = same_stats[same_stats['turn_number'] == MAX_TURN]['mean_clue_util'].values[0] if MAX_TURN in same_stats['turn_number'].values else same_stats['mean_clue_util'].values[-1]
    diff_t1 = diff_stats[diff_stats['turn_number'] == 1]['mean_clue_util'].values[0]
    diff_t6 = diff_stats[diff_stats['turn_number'] == MAX_TURN]['mean_clue_util'].values[0] if MAX_TURN in diff_stats['turn_number'].values else diff_stats['mean_clue_util'].values[-1]
    
    same_drop = same_t1 - same_t6
    diff_drop = diff_t1 - diff_t6
    
    log_print(f"   Same Persona: Turn 1 ({same_t1:.3f}) → Turn {MAX_TURN} ({same_t6:.3f}), Drop = {same_drop:.3f}")
    log_print(f"   Different Persona: Turn 1 ({diff_t1:.3f}) → Turn {MAX_TURN} ({diff_t6:.3f}), Drop = {diff_drop:.3f}")

# 3. Significant differences
log_print(f"\n3. TURNS WITH SIGNIFICANT DIFFERENCES (p < 0.05):")
sig_turns = test_results_df[test_results_df['sig'] != '(n.s.)']
if len(sig_turns) > 0:
    for _, row in sig_turns.iterrows():
        log_print(f"   Turn {int(row['turn'])}: Same ({row['same_mean']:.3f}) vs Diff ({row['diff_mean']:.3f}), p = {row['p_value']:.4f} {row['sig']}")
else:
    log_print("   No turns showed statistically significant differences.")

# 4. Interpretation
log_print(f"\n=== INTERPRETATION ===")
log_print("\n'Low-Hanging Fruit' Hypothesis Test:")

# Check if the hypothesis is supported
if len(test_results_df) > 0:
    early_sig = test_results_df[test_results_df['turn'] <= 2]['sig'].tolist()
    late_sig = test_results_df[test_results_df['turn'] >= 4]['sig'].tolist()
    
    early_has_sig = any(s != '(n.s.)' for s in early_sig)
    late_has_sig = any(s != '(n.s.)' for s in late_sig)
    
    if not early_has_sig and late_has_sig:
        log_print("✓ SUPPORTED: No significant difference in early turns, but significant")
        log_print("  divergence in later turns. This is consistent with the hypothesis that")
        log_print("  Same Persona pairs maintain their 'shared protocol' when the game gets hard.")
    elif early_has_sig and late_has_sig:
        log_print("~ PARTIALLY SUPPORTED: Same Persona pairs show advantage throughout,")
        log_print("  including early turns. This suggests the benefit is not just about")
        log_print("  late-game resilience but consistent throughout gameplay.")
    elif not early_has_sig and not late_has_sig:
        log_print("✗ NOT SUPPORTED: No significant differences found at any turn.")
        log_print("  The data does not show a clear divergence pattern between cohorts.")
    else:
        log_print("? INCONCLUSIVE: Unexpected pattern - early difference without late difference.")

log_print("\n" + "=" * 80)
log_print("END OF TURN-LEVEL DYNAMICS ANALYSIS")
log_print("=" * 80)